## Imports

In [1]:
import os
import pandas as pd
import numpy as np

## Constants

In [2]:
str_dirname_output = './output'
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

## Read

In [3]:
str_filename = 'df_ecnl.csv'
df = pd.read_csv(f'./input/df_ecnl.csv')
df.head(10)

,bigAccountId,bigDealerId,week_of_month,application_date,dtmFunded,dtmNonAccrual,intterm,open_bk_type,first_pmt_date,paymentsmade,...,LoanStatus,ChargeOff_Severity,GEN11_ExpectedNetLoss,GEN11_ECNL,GEN12_ExpectedNetLoss,GEN12_ECNL,GEN12_ECNL_mod,DL_ExpectedNetLoss,DL_ECNL,origination_model
0,7622142,2836,4,44:51.2,00:00.0,NaN,72,13.0,00:00.0,2.0,...,A,NaN,2462.247980,0.095872,1898.025622,0.073903,0.051137,NaN,NaN,gen11
1,7794020,939,4,54:06.9,00:00.0,NaN,72,7.0,00:00.0,NaN,...,A,NaN,1203.251490,0.087003,1190.126820,0.086054,0.059545,NaN,NaN,gen11
2,7844199,2947,4,01:16.3,00:00.0,NaN,72,13.0,00:00.0,NaN,...,A,NaN,1621.960426,0.073042,2405.094491,0.108309,0.074945,NaN,NaN,gen11
3,7893285,7480,4,32:15.1,00:00.0,NaN,72,7.0,00:00.0,1.0,...,A,NaN,2026.329035,0.118288,9035.534923,0.527454,0.223498,NaN,NaN,gen11
4,7897599,8308,4,29:15.2,00:00.0,NaN,72,NaN,00:00.0,1.0,...,A,NaN,2632.401450,0.118125,10632.918500,0.477136,0.202176,NaN,NaN,gen11
5,7909310,4250,4,49:24.2,00:00.0,NaN,72,7.0,00:00.0,NaN,...,A,NaN,4535.329257,0.147052,2498.144428,0.080999,0.056047,NaN,NaN,gen12
6,7910522,6845,4,50:07.8,00:00.0,NaN,48,NaN,00:00.0,NaN,...,A,NaN,1646.546137,0.097343,3230.388777,0.190979,0.123076,NaN,NaN,gen11
7,7916574,3171,4,19:19.9,00:00.0,NaN,72,NaN,00:00.0,NaN,...,A,NaN,1291.481648,0.090462,4908.249861,0.343799,0.145678,NaN,NaN,gen11
8,7919227,1727,4,47:33.3,00:00.0,NaN,66,7.0,00:00.0,NaN,...,A,NaN,1572.411456,0.085184,6532.713936,0.353904,0.149959,NaN,NaN,gen11
9,7921215,8116,4,00:03.6,00:00.0,NaN,72,NaN,00:00.0,NaN,...,A,NaN,4513.352839,0.184530,3985.168508,0.162935,0.105003,NaN,NaN,gen12


In [4]:
df_ecnl = df[['bigAccountId', 'AmtFinanced', 'mnyDiscount', 'mnyReserve', 'fltAPRCurrent', 'Tier', 'GEN11_ECNL']]
df_ecnl.head(10)

,bigAccountId,AmtFinanced,mnyDiscount,mnyReserve,fltAPRCurrent,Tier,GEN11_ECNL
0,7622142,25682.66,3278,941.47,0.2500,B,0.095872
1,7794020,13830.00,505,872.42,0.2524,B,0.087003
2,7844199,22205.86,539,1397.59,0.2524,B,0.073042
3,7893285,17130.47,1390,0.00,0.2324,B,0.118288
4,7897599,22284.88,928,726.77,0.2555,B,0.118125
5,7909310,30841.67,1378,84.17,0.2100,A,0.147052
6,7910522,16914.89,500,645.12,0.2424,B,0.097343
7,7916574,14276.51,219,900.14,0.2524,B,0.090462
8,7919227,18459.00,149,368.21,0.2395,B,0.085184
9,7921215,24458.64,928,928.87,0.2995,B,0.184530


In [16]:
# formulas for each tier category to calc theo_rev
# each named var here is the default value, but in the app, each var corresponds to a user-input value
values_A1 = (.0807 + 1.11 * (df['GEN12_ECNL'])/1.5209) * .275 + (1-(.0807 + 1.11 * (df['GEN12_ECNL'])/1.5209)) * .07 + .0564 + (df['GEN12_ECNL'])/2.3) - ((df['GEN12_ECNL']/1.5209) * 0.008)
values_A = (.0807 + 1.11 * (df['GEN12_ECNL'])) * .275 + (1-(.0807 + 1.11 * (df['GEN12_ECNL']))) * .07 + .0603 + ((df['GEN12_ECNL'])*1.5209/2.3)
values_B = (.0807 + 1.11 * (df['GEN12_ECNL'])) * .275 + (1-(.0807 + 1.11 * (df['GEN12_ECNL']))) * .07 + .0639 + ((df['GEN12_ECNL'])*1.633/2.3)
values_C = (.0807 + 1.11 * (df['GEN12_ECNL'])) * .275 + (1-(.0807 + 1.11 * (df['GEN12_ECNL']))) * .07 + .0639 + ((df['GEN12_ECNL'])*1.5209/2.3)
values_D = (.0807 + 1.11 * (df['GEN12_ECNL'])) * .275 + (1-(.0807 + 1.11 * (df['GEN12_ECNL']))) * .07 + .0639 + ((df['GEN12_ECNL'])*1.5209/2.3)

# incorporate dynamic formula to calculate the theo_rev for each tier category
    # originally did not work since i had to distinguish gen11/12 which in that case had to use the numpy below (not fun)
for i in df.Tier:
    if i == 'A1':
        df['GEN11_Theo_Rev'] = values_A1
    elif i == 'A':
        df['GEN11_Theo_Rev'] = values_A
    elif i == 'B':
        df['GEN11_Theo_Rev'] = values_B
    elif i == 'C' or 'D':
        df['GEN11_Theo_Rev'] = values_C

# # THE FOLLOWING IS NOT NEEDED SINCE ONLY GEN12 AS OF 7/12
# # initialize the column with default values
# df['GEN11_Theo_Rev'] = np.nan

# # use np.where to assign values based on conditions
# df['GEN11_Theo_Rev'] = np.where(
#     (df['Tier'] == 'A1') & (df['bigAccountId'] % 5 != 0), values_A1,
#     np.where(
#         (df['Tier'] == 'A') & (df['bigAccountId'] % 5 != 0), values_A,
#         np.where(
#             (df['Tier'] == 'B') & (df['bigAccountId'] % 5 != 0), values_B,
#             np.where(
#                 ((df['Tier'] == 'C') | (df['Tier'] == 'D')) & (df['bigAccountId'] % 5 != 0), values_C, 
#                 df['GEN11_Theo_Rev']
#                    ... add rest ...
#             )
#         )
#     )
# )

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [10]:
df['GEN11_Theo_Rev'].head

<bound method NDFrame.head of 0     0.240328
1          NaN
2     0.218924
3          NaN
4     0.261192
5          NaN
6     0.241707
7     0.235256
8     0.230308
9          NaN
10    0.288458
11    0.314879
12    0.277575
13    0.312624
14    0.259980
15    0.251898
16    0.270092
17    0.265422
18    0.250166
19    0.244716
20         NaN
21    0.301605
22    0.294271
23    0.248966
24    0.257474
25    0.228516
26    0.222197
27    0.200766
28    0.231219
29    0.194687
30    0.276336
31    0.291892
32    0.310175
33    0.317030
34    0.307939
35    0.300288
36    0.249386
37    0.245732
38    0.271095
39    0.285851
40    0.272627
41    0.272236
42    0.223133
43    0.276812
44    0.307305
45    0.304525
46    0.236729
47    0.219141
Name: GEN11_Theo_Rev, dtype: float64>

In [11]:
df['Actual_Rev'] = df['OriginalInterestRate'] + ((df['mnyDiscount'] - df['mnyReserve']) / df['AmtFinanced'])/2.3
df['Actual_Rev'].head

<bound method NDFrame.head of 0     0.289555
1     0.240849
2     0.235589
3     0.267679
4     0.259426
5     0.228239
6     0.238670
7     0.231656
8     0.234337
9     0.299485
10    0.288795
11    0.300030
12    0.278764
13    0.309951
14    0.259648
15    0.243933
16    0.272237
17    0.248392
18    0.239278
19    0.247608
20    0.300010
21    0.288996
22    0.295974
23    0.248997
24    0.251196
25    0.228779
26    0.236993
27    0.199738
28    0.236098
29    0.191750
30    0.276115
31    0.306343
32    0.322593
33    0.301951
34    0.305149
35    0.305791
36    0.254139
37    0.241162
38    0.265240
39    0.293767
40    0.278577
41    0.274802
42    0.243414
43    0.274578
44    0.309416
45    0.305701
46    0.237155
47    0.236438
Name: Actual_Rev, dtype: float64>

In [12]:
df['Rev_Diff'] = df['Actual_Rev'] - df['GEN11_Theo_Rev']
df['Rev_Diff'].head

<bound method NDFrame.head of 0     0.049227
1          NaN
2     0.016665
3          NaN
4    -0.001766
5          NaN
6    -0.003038
7    -0.003600
8     0.004029
9          NaN
10    0.000337
11   -0.014849
12    0.001189
13   -0.002673
14   -0.000333
15   -0.007965
16    0.002145
17   -0.017030
18   -0.010888
19    0.002892
20         NaN
21   -0.012609
22    0.001703
23    0.000031
24   -0.006278
25    0.000263
26    0.014796
27   -0.001028
28    0.004879
29   -0.002937
30   -0.000221
31    0.014450
32    0.012418
33   -0.015079
34   -0.002790
35    0.005503
36    0.004753
37   -0.004571
38   -0.005855
39    0.007916
40    0.005950
41    0.002566
42    0.020282
43   -0.002234
44    0.002111
45    0.001176
46    0.000426
47    0.017297
Name: Rev_Diff, dtype: float64>

In [13]:
df = df.sort_values(by=['Rev_Diff'], ascending=True)
df

,bigAccountId,bigDealerId,week_of_month,application_date,dtmFunded,dtmNonAccrual,intterm,open_bk_type,first_pmt_date,paymentsmade,...,GEN11_ECNL,GEN12_ExpectedNetLoss,GEN12_ECNL,GEN12_ECNL_mod,DL_ExpectedNetLoss,DL_ECNL,origination_model,GEN11_Theo_Rev,Actual_Rev,Rev_Diff
17,7935253,2612,4,23:13.7,00:00.0,NaN,72,NaN,00:00.0,NaN,...,0.122637,2709.543923,0.078486,0.050580,NaN,NaN,gen11,0.265422,0.248392,-0.017030
33,7947387,7980,4,44:58.4,00:00.0,NaN,66,NaN,00:00.0,NaN,...,0.187426,13270.548020,0.707615,0.299837,NaN,NaN,gen11,0.317030,0.301951,-0.015079
11,7926947,8171,4,10:05.1,00:00.0,NaN,72,7.0,00:00.0,NaN,...,0.185006,2492.094279,0.138273,0.095678,NaN,NaN,gen11,0.314879,0.300030,-0.014849
21,7943081,342,4,09:52.1,00:00.0,NaN,72,NaN,00:00.0,NaN,...,0.170072,11000.536980,0.374296,0.158600,NaN,NaN,gen11,0.301605,0.288996,-0.012609
18,7935981,3615,4,25:00.5,00:00.0,NaN,72,13.0,00:00.0,NaN,...,0.106365,2749.683191,0.118209,0.081795,NaN,NaN,gen11,0.250166,0.239278,-0.010888
15,7934934,8406,4,23:01.3,00:00.0,NaN,72,NaN,00:00.0,NaN,...,0.108212,2975.059175,0.135765,0.093942,NaN,NaN,gen11,0.251898,0.243933,-0.007965
24,7944606,6502,4,23:39.5,00:00.0,NaN,48,NaN,00:00.0,NaN,...,0.114160,16642.345660,0.517747,0.219385,NaN,NaN,gen11,0.257474,0.251196,-0.006278
38,7956801,2687,4,13:28.9,00:00.0,NaN,72,7.0,00:00.0,NaN,...,0.128688,3018.478647,0.127215,0.088026,NaN,NaN,gen11,0.271095,0.265240,-0.005855
37,7954554,6641,4,28:02.7,00:00.0,NaN,72,NaN,00:00.0,NaN,...,0.101636,8207.749474,0.433302,0.183602,NaN,NaN,gen11,0.245732,0.241162,-0.004571
7,7916574,3171,4,19:19.9,00:00.0,NaN,72,NaN,00:00.0,NaN,...,0.090462,4908.249861,0.343799,0.145678,NaN,NaN,gen11,0.235256,0.231656,-0.003600


In [ ]:
str_filename = 'df_theo_rev.csv'
df.to_csv(f'./output/{str_filename}')